[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C35_Speech_Audio_Course/04_tts/04_tts.ipynb)

# 04 · TTS 与声码器（用 numpy 从零实现）

目标：把 **iSTFT（overlap-add）与 Griffin-Lim 相位重建** 用 numpy 从零实现，
并对拍核心不变量——**有真相位时 iSTFT∘STFT ≈ 恒等**、**Griffin-Lim 的谱收敛度随迭代单调下降**。

路线：STFT/iSTFT 往返 → 随机相位的灾难 → Griffin-Lim 迭代 → 谱收敛单调(对拍) → 收敛曲线 → GL vs 随机 → ✏️ 练习 → 📖 答案 → 🧪 配置胶囊。

> 心智模型：谱图丢了相位；声码器要把相位**猜回来**。Griffin-Lim 在时域↔频域间反复投影逼出自洽相位。

## 1 · STFT 与 iSTFT：有真相位时无损往返

先建好 STFT（复谱）与 iSTFT（overlap-add 逆变换）。
**关键对拍**：给一个真实波形，`iSTFT(STFT(x)) ≈ x`（误差极小）——这验证 OLA 与窗归一写对了。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def hann_window(N):
    return 0.5 - 0.5*np.cos(2*np.pi*np.arange(N)/N)

def stft(x, frame_len=256, hop=128):
    w = hann_window(frame_len)
    T = 1 + (len(x) - frame_len)//hop
    idx = np.arange(frame_len)[None,:] + hop*np.arange(T)[:,None]
    return np.fft.rfft(x[idx]*w[None,:], axis=1)

def istft(S, frame_len=256, hop=128):
    '''overlap-add 逆变换：每帧 irfft 回时域，按 hop 重叠相加，再除以窗能量。'''
    w = hann_window(frame_len)
    frames = np.fft.irfft(S, n=frame_len, axis=1)   # (T, frame_len)
    T = frames.shape[0]
    n = (T-1)*hop + frame_len
    out = np.zeros(n); wsum = np.zeros(n)
    for i in range(T):
        out[i*hop:i*hop+frame_len]  += frames[i]*w     # 合成时再乘一次窗(标准做法)
        wsum[i*hop:i*hop+frame_len] += w*w
    wsum[wsum < 1e-10] = 1e-10
    return out/wsum

# 造一个有结构的波形
t = np.arange(4096)/256.0
x = (np.sin(2*np.pi*5*t) + 0.5*np.sin(2*np.pi*12*t)) * np.hanning(4096)
S = stft(x)
x_rec = istft(S)
# 比较重叠充分的中间段(避开两端边界)
L = min(len(x), len(x_rec))
a, b = 256, L-256
err = np.max(np.abs(x[a:b] - x_rec[a:b]))
print(f'iSTFT(STFT(x)) 中段最大误差 = {err:.2e}')
assert err < 1e-6, 'OLA 往返应近似无损'
print('✅ 有真相位时 iSTFT∘STFT ≈ 恒等（overlap-add + 窗归一正确）')

## 2 · 随机相位的灾难：为什么不能瞎配

只有幅度谱时，若直接配**随机相位**再 iSTFT，重建波形的 STFT 幅度会与目标差很远（帧间打架）。
用谱收敛度 `SC = ‖|STFT(x̂)| - A‖ / ‖A‖` 度量——随机相位的 SC 很大。

In [ ]:
def spectral_convergence(x_hat, A, frame_len=256, hop=128):
    '''重建信号的 STFT 幅度 与 目标幅度 A 的相对差。'''
    S = stft(x_hat, frame_len, hop)
    m = min(S.shape[0], A.shape[0])
    return float(np.linalg.norm(np.abs(S[:m]) - A[:m]) / (np.linalg.norm(A[:m]) + 1e-10))

A = np.abs(stft(x))            # 目标幅度谱（丢掉相位）
# 随机相位直接 iSTFT
rand_phase = np.exp(2j*np.pi*rng.random(A.shape))
x_rand = istft(A * rand_phase)
sc_rand = spectral_convergence(x_rand, A)
# 对照：用真相位(作弊)的 SC 应≈0
sc_true = spectral_convergence(istft(stft(x)), A)
print(f'随机相位 谱收敛度 = {sc_rand:.4f} (大 -> 帧间不自洽)')
print(f'真相位   谱收敛度 = {sc_true:.4f} (≈0 -> 自洽)')
assert sc_rand > sc_true + 0.1, '随机相位应明显更差'
print('✅ 随机相位 iSTFT 不自洽（SC 大）—— 相位必须好好猜，不能瞎配')

## 3 · Griffin-Lim：交替投影恢复相位 ★

`S = A·e^{iφ0}` (随机初相)；每轮 `x=iSTFT(S); S = A·e^{i∠STFT(x)}`。

两步投影：iSTFT 强制成合法时域信号（OLA 调和帧间）、换幅度强制 |S|=A。返回波形 + 每轮谱收敛度。

In [ ]:
def griffin_lim(A, n_iter=60, frame_len=256, hop=128, seed=0):
    '''从幅度谱 A 重建波形。返回 (波形, 每轮谱收敛度 list)。'''
    g = np.random.default_rng(seed)
    phase = np.exp(2j*np.pi*g.random(A.shape))
    S = A * phase
    history = []
    x_hat = istft(S, frame_len, hop)
    for _ in range(n_iter):
        x_hat = istft(S, frame_len, hop)
        Snew = stft(x_hat, frame_len, hop)
        m = min(Snew.shape[0], A.shape[0])
        phase = np.exp(1j*np.angle(Snew[:m]))   # 取当前信号的自然相位
        S = A[:m] * phase                        # 幅度强制 = A，只换相位
        history.append(spectral_convergence(x_hat, A, frame_len, hop))
    return istft(S, frame_len, hop), history

x_gl, hist = griffin_lim(A, n_iter=60)
print(f'Griffin-Lim 60 次迭代:')
print(f'  初始(迭代1) 谱收敛度 = {hist[0]:.4f}')
print(f'  最终(迭代60)谱收敛度 = {hist[-1]:.4f}')
assert hist[-1] < hist[0], 'GL 应改善谱收敛度'
assert hist[-1] < sc_rand, 'GL 应远好于随机相位'
print('✅ Griffin-Lim 把谱收敛度从随机水平大幅降低（逼出自洽相位）')

## 4 · 核心不变量：谱收敛度随迭代单调下降 ★

交替投影保证目标 `‖|S|-A‖` 单调不增。验证 `hist` **单调下降**（允许浮点级微小波动）。

In [ ]:
# ★ 单调性验证（交替投影理论保证不增；留极小数值容差）
violations = sum(1 for i in range(1, len(hist)) if hist[i] > hist[i-1] + 1e-9)
print(f'非单调违例数 = {violations} / {len(hist)-1}')
assert violations == 0, 'Griffin-Lim 谱收敛度应单调不增'
# 收敛：后期改善变小
early_improve = hist[0] - hist[10]
late_improve = hist[-11] - hist[-1]
print(f'前10轮改善 = {early_improve:.4f}, 后10轮改善 = {late_improve:.4f}')
assert late_improve < early_improve, '后期边际改善应变小（趋于收敛）'
print('✅ ★ 谱收敛度严格单调下降，且边际改善递减（趋于局部最优）')

## 5 · 收敛曲线：迭代次数怎么选

把谱收敛度随迭代画成曲线（文本版），看它何时趋平——这决定实践中迭代次数的选择。

In [ ]:
# 文本版收敛曲线
print('迭代   谱收敛度   ' + '进度条')
maxv = max(hist)
for k in [0, 1, 2, 4, 9, 19, 39, 59]:
    v = hist[k]
    bar = '#' * int(40*v/maxv)
    print(f'{k+1:>4}   {v:.4f}    {bar}')
# 量化『趋平』：到第 30 轮已达成大部分改善
total = hist[0] - hist[-1]
at30 = hist[0] - hist[min(29, len(hist)-1)]
frac = at30/total if total > 0 else 1.0
print(f'\n前 30 轮完成了 {frac:.0%} 的总改善')
assert frac > 0.7, '大部分改善应在前几十轮完成'
print('✅ 收敛曲线前陡后平：实践中 30–100 次迭代即可')

## 6 · Griffin-Lim 确实有效：多 seed 对照

对多个随机初相，验证 Griffin-Lim 的最终谱收敛度**始终远优于**随机相位（鲁棒、确有效）。

In [ ]:
results = []
for seed in range(5):
    _, h = griffin_lim(A, n_iter=50, seed=seed)
    rp = np.exp(2j*np.pi*np.random.default_rng(seed).random(A.shape))
    sc_r = spectral_convergence(istft(A*rp), A)
    results.append((seed, h[-1], sc_r))
    print(f'seed={seed}: GL最终SC={h[-1]:.4f}  随机SC={sc_r:.4f}  改善={sc_r/h[-1]:.1f}x')
for seed, gl, rd in results:
    assert gl < rd, f'seed {seed}: GL 应优于随机'
mean_gl = np.mean([r[1] for r in results])
mean_rd = np.mean([r[2] for r in results])
print(f'\n平均: GL={mean_gl:.4f} vs 随机={mean_rd:.4f}')
print('✅ 多 seed 一致：Griffin-Lim 鲁棒地优于随机相位')

---
## ✏️ 练习 1：实现 iSTFT（overlap-add）

实现 `my_istft(S, N, H)`：每帧 `irfft` 回时域、乘合成窗、按 hop 重叠相加、除以窗能量。
验证：对一个完整复谱，`my_istft(stft(x))` 在中段 ≈ x。

In [ ]:
def my_istft(S, N=256, H=128):
    # TODO:
    #  1) frames = np.fft.irfft(S, n=N, axis=1); w=hann_window(N)
    #  2) out=zeros(n), wsum=zeros(n), n=(T-1)*H+N
    #  3) 逐帧: out[i*H:i*H+N]+=frames[i]*w; wsum 同样 += w*w
    #  4) 返回 out/clip(wsum)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
t = np.arange(3072)/256.0
xx = np.sin(2*np.pi*7*t)*np.hanning(3072)
rec = my_istft(stft(xx))
L = min(len(xx), len(rec))
assert np.max(np.abs(xx[256:L-256] - rec[256:L-256])) < 1e-6
print('✅ 练习 1 通过：iSTFT overlap-add 往返近似无损')

## ✏️ 练习 2：谱收敛度

实现 `spec_conv(x_hat, A, N, H)`：返回 `‖|STFT(x_hat)| - A‖ / ‖A‖`（注意对齐帧数）。

In [ ]:
def spec_conv(x_hat, A, N=256, H=128):
    # TODO: S=stft(x_hat); m=min(帧数); 返回 norm(|S[:m]|-A[:m])/norm(A[:m])
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
A2 = np.abs(stft(xx))
# 真相位 -> ≈0
assert spec_conv(my_istft(stft(xx)), A2) < 1e-3
# 随机相位 -> 较大
rp = np.exp(2j*np.pi*rng.random(A2.shape))
assert spec_conv(my_istft(A2*rp), A2) > 0.1
print('✅ 练习 2 通过：谱收敛度区分自洽(真相位)与不自洽(随机)')

## ✏️ 练习 3：Griffin-Lim 一轮迭代

实现 `gl_step(S, A, N, H)`：执行**一轮** Griffin-Lim——`x=istft(S); 取∠STFT(x); 返回 A·e^{i相位}`。
（注意对齐帧数。）这是 Griffin-Lim 的核心，循环调用它即可。

In [ ]:
def gl_step(S, A, N=256, H=128):
    # TODO: x=istft(S,N,H); Snew=stft(x,N,H); m=min(帧数);
    #       phase=exp(1j*angle(Snew[:m])); 返回 A[:m]*phase
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
A3 = np.abs(stft(xx))
S = A3 * np.exp(2j*np.pi*rng.random(A3.shape))   # 随机初相
sc_before = spec_conv(my_istft(S), A3)
for _ in range(30):
    S = gl_step(S, A3)
sc_after = spec_conv(my_istft(S), A3)
# 幅度应始终 = A（只换相位）
assert np.allclose(np.abs(S), A3[:S.shape[0]], atol=1e-9), '每步应强制 |S|=A'
assert sc_after < sc_before, '30 轮后谱收敛度应改善'
print(f'30 轮 gl_step: SC {sc_before:.4f} -> {sc_after:.4f}')
print('✅ 练习 3 通过：一轮 GL 正确（保幅度、换相位、改善自洽性）')

## ✏️ 练习 4：完整 Griffin-Lim + 单调性

用练习 3 的 `gl_step` 组装 `my_griffin_lim(A, n_iter, N, H, seed)`：随机初相，迭代 n_iter 轮，
返回 `(波形, 每轮谱收敛度 list)`。验证谱收敛度单调不增。

In [ ]:
def my_griffin_lim(A, n_iter=50, N=256, H=128, seed=0):
    # TODO: 随机初相 S=A*e^{iφ}; 循环 n_iter 轮: 记 spec_conv(istft(S),A);
    #       S=gl_step(S,A); 最后返回 (istft(S), history)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
wav, h = my_griffin_lim(A3, n_iter=40, seed=1)
assert len(h) == 40
viol = sum(1 for i in range(1, len(h)) if h[i] > h[i-1] + 1e-9)
assert viol == 0, '谱收敛度应单调不增'
assert h[-1] < h[0]
print(f'40 轮: SC {h[0]:.4f} -> {h[-1]:.4f}, 单调违例={viol}')
print('✅ 练习 4 通过：完整 Griffin-Lim，谱收敛度单调下降')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def my_istft(S, N=256, H=128):
    w = hann_window(N)
    frames = np.fft.irfft(S, n=N, axis=1)
    T = frames.shape[0]; n = (T-1)*H + N
    out = np.zeros(n); wsum = np.zeros(n)
    for i in range(T):
        out[i*H:i*H+N]  += frames[i]*w
        wsum[i*H:i*H+N] += w*w
    wsum[wsum < 1e-10] = 1e-10
    return out/wsum

In [ ]:
# 练习 2 参考答案
def spec_conv(x_hat, A, N=256, H=128):
    S = stft(x_hat, N, H)
    m = min(S.shape[0], A.shape[0])
    return float(np.linalg.norm(np.abs(S[:m]) - A[:m]) / (np.linalg.norm(A[:m]) + 1e-10))

In [ ]:
# 练习 3 参考答案
def gl_step(S, A, N=256, H=128):
    x = my_istft(S, N, H)
    Snew = stft(x, N, H)
    m = min(Snew.shape[0], A.shape[0])
    return A[:m] * np.exp(1j*np.angle(Snew[:m]))

In [ ]:
# 练习 4 参考答案
def my_griffin_lim(A, n_iter=50, N=256, H=128, seed=0):
    g = np.random.default_rng(seed)
    S = A * np.exp(2j*np.pi*g.random(A.shape))
    history = []
    for _ in range(n_iter):
        history.append(spec_conv(my_istft(S, N, H), A, N, H))
        S = gl_step(S, A, N, H)
    return my_istft(S, N, H), history

---
## 🧪 真实数据胶囊：TTS 声码器的真实 STFT 配置

真实 TTS（如 Tacotron 2 + 声码器）用一组**公开的真实** STFT/梅尔参数。我们用这些真实数值，
对一段合成『语音样的』信号跑完整的 **梅尔谱 → Griffin-Lim → 波形** 流程，验证它在真实配置下也单调收敛。

（纯 numpy/标准库，无需联网、无需 TTS 包。）

In [ ]:
# Tacotron 2 / 常见声码器的真实 STFT 配置（22.05 kHz 体系）
TTS_CFG = dict(sample_rate=22050, n_fft=1024, hop=256, win=1024)

def synth_speechlike(sr, dur=0.5, f0=120.0):
    '''合成一个『语音样』信号：基频 + 谐波(模拟浊音)，带幅度包络。'''
    t = np.arange(int(sr*dur))/sr
    sig = sum((1.0/k)*np.sin(2*np.pi*f0*k*t) for k in range(1, 8))   # 谐波梳
    env = 0.5*(1 - np.cos(2*np.pi*np.arange(len(t))/len(t)))         # 包络
    return sig*env

cfg = TTS_CFG
x = synth_speechlike(cfg['sample_rate'])
A_real = np.abs(stft(x, cfg['n_fft'], cfg['hop']))
wav, hist = griffin_lim(A_real, n_iter=50, frame_len=cfg['n_fft'], hop=cfg['hop'], seed=0)
print(f"真实配置 {cfg['sample_rate']}Hz, n_fft={cfg['n_fft']}, hop={cfg['hop']}")
print(f'谱收敛度: 迭代1={hist[0]:.4f} -> 迭代50={hist[-1]:.4f}')
viol = sum(1 for i in range(1,len(hist)) if hist[i] > hist[i-1]+1e-9)
assert viol == 0 and hist[-1] < hist[0]
print('✅ 真实 TTS STFT 配置下 Griffin-Lim 也单调收敛')

**🧪 胶囊练习**：实现 `stft_time_resolution_ms(hop, sr)` 与 `stft_freq_resolution_hz(n_fft, sr)`：
算出真实配置下的时间分辨率（帧移对应毫秒）与频率分辨率（频点间隔 Hz）——这正是模块 01 时频权衡的落地。

In [ ]:
def stft_time_resolution_ms(hop, sr):
    # TODO: hop/sr*1000
    raise NotImplementedError

def stft_freq_resolution_hz(n_fft, sr):
    # TODO: sr/n_fft
    raise NotImplementedError

In [ ]:
# 自测
tr = stft_time_resolution_ms(256, 22050)
fr = stft_freq_resolution_hz(1024, 22050)
assert abs(tr - 256/22050*1000) < 1e-6
assert abs(fr - 22050/1024) < 1e-6
print(f'时间分辨率 = {tr:.1f} ms/帧；频率分辨率 = {fr:.1f} Hz/频点')
print('✅ 胶囊练习通过：真实配置的时频分辨率')

In [ ]:
# 📖 胶囊参考答案
def stft_time_resolution_ms(hop, sr):
    return hop/sr*1000

def stft_freq_resolution_hz(n_fft, sr):
    return sr/n_fft

### 小结
- 从谱图还原波形是**逆问题**：缺的是**相位**；有完整复谱时 iSTFT(overlap-add) 平凡无损。
- 随机相位 iSTFT = 灾难（帧间不自洽）；声码器必须把相位**猜回来**。
- **Griffin-Lim**：时域↔频域**交替投影**，保幅度换相位；谱收敛度**单调下降**（局部最优）。
- 音质上限是结构性的（无真实语音先验）；**神经声码器**(WaveNet/HiFi-GAN)用数据学相位突破之。
- 声码器 ≈ 神经编解码解码器（同源）；分析丢的相位正是生成要还的。

下一站：**模块 05 · 语音 LLM 与流式** —— 把离散 token 纳入语言模型，生成方向的终点。